# Tutorial: Making bulk mosaic cutouts using Cutana

<br>
<br>

<img style="position: absolute; right: 60px; top: 40px; height: 160px; width: 160px" src="https://datalabs.esa.int/datalab-icon/c11e47af-7ce9-4281-b3b5-475a180a256f" /> 

**Notebook author(s):** Kristin Anett Remmelgas, Antonio La Marca

**Last modified:** 20.03.2026

**Cutana software author(s):** Pablo Gómez, David O'Ryan, Laslo Ruhberg

**Notebook summary**

This example notebook demonstrates a simple example of how to use the Cutana software installed in this datalab to make bulk cutouts of Euclid mosaics. Cutana is a high-performance Python pipeline for creating astronomical image cutouts from large FITS tile collections, designed for efficient processing of ESA Datalabs and other astronomical datasets. First in this notebook we demonstrate what Cutana can do on an example source catalogue. Then in the second part of the notebook we look at how to run Cutana without the user interfact. Here is an overview of what is covered:

1. Cutana Demo
2. Running the Cutana without the user interface


**Useful links:**

* [Cutana documentation](../home/Cutana/README.md)
* [Generate Cutana Input](/Cutana/Input_file_for_bulk_cutouts.ipynb) - Tutorial notebook on how to generate the input file for Cutana.

**Running the notebook:** 

* This notebook has **read-only permissions**. You can still run and edit cells but if you would like to save your changes then you have to save the notebook to a different location - your workspace for example. 
* Cutana part of the notebook is intended to be run using the **cutana kernel**
***

In [ ]:
#making a folder for the outputs (if not made already)
import os
cutana_ouput_folder= '/media/user/example_notebook_outputs/cutana_output'
if not os.path.exists(cutana_ouput_folder):
    os.makedirs(cutana_ouput_folder)

## 1. Cutana demo

Cutana is installed in a separate environment so **please make sure the cutana kernel is selected on the top right** (instead of EUCLID_TOOLS).

After running the cell below a user interface will show up to prompt you to

1. First select a source cataogue - a demo input file (cutana_input.csv) to use is available in the Cutana folder
2. Then select output directory for the cutouts. 
3. Then you can click on Start Cutana, select some more parameters for your cutouts and click on Start Cutout Creation.

Note that you can click on the red Help button on the top right to see the Cutana documentation.

In [ ]:
#this notebooks need a version of drizzle installed in the cutana kernel
%pip install drizzle

In [ ]:
#Update Cuatana to the latest version
%pip install --upgrade cutana

In [ ]:
# UI-based start
import cutana_ui
cutana_ui.start()

In [ ]:
# look at the zarr files directly
import zarr
from matplotlib import pyplot as plt
from datetime import datetime
import numpy as np
import glob

# open file for the first batch of today
today = datetime.today().strftime('%Y%m%d')
all_todays_zarr_files = glob.glob(cutana_ouput_folder + '/' + today + '*/*/images.zarr')
file = zarr.open(all_todays_zarr_files[0], mode="r",)

# display 16 random images in a grid
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ax in axes.flatten():
    image = file["images"][np.random.randint(0, file["images"].shape[0])]
    #if using multi band images
    ax.imshow(image[:,:,0])
    #if using single band images
    #ax.imshow(image)
    ax.axis("off")
plt.tight_layout()
plt.show()

The user can generate their own cutana input file from MER queries or an source input file. We refer the users to [Generate Cutana Input](Input_file_for_bulk_cutouts.ipynb) for an example on how to generate such file.

***

## 2. Running Cutana without the user interface

Make sure you are using the **cutana kernel** to run cutana.

In [ ]:
from cutana import get_default_config, Orchestrator

In [ ]:
%%time

# Make sure the paths point to correct location here
config = get_default_config()
config.source_catalogue = "/media/team_workspaces/Euclid-Consortium/Example_Notebooks/Cutana/cutana_input_idr1.csv"
# config.source_catalogue = '/media/user/example_notebook_outputs/cutana_input-comparison.csv'
config.output_dir = "/media/user/example_notebook_outputs/cutana_output"

#set parameters
config.output_format = "zarr"
config.target_resolution = 200  # Target resolution in pixels for the created cutouts
config.console_log_level = "WARNING"  # Set to INFO for more insight on what's going on
# Extensions to process , here we only do VIS, could be e.g. ["VIS", "NIR-H"], has to match fits order
# 1 output channel for VIS, details explained below
config.selected_extensions = ["VIS"]
# Manually define weights for channel blending, order has to match selected_extensions and fits
config.channel_weights = {"VIS": [1.0],}

# Process cutouts
orchestrator = Orchestrator(config)
results = orchestrator.run()

In [ ]:
results

In [ ]:
# Get sources info for a tile
import pandas as pd

# Each created batch / zarr has their metadata saved in a parquet file
metadata = pd.read_parquet('/media/user/example_notebook_outputs/cutana_output/source_to_zarr_mapping.parquet')
metadata.head()

In [ ]:
Euclid.logout()